In [0]:
 %load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

# Leer desde la capa de bronce

In [0]:
from pyspark.sql.functions import col, when,trim,create_map, regexp_replace,lit,to_date,rlike, expr,coalesce, lower,broadcast, max, min, avg, stddev,row_number, sum as suma
from itertools import chain
from pyspark.sql.types import StringType
from pyspark.sql.window import Window
from clean import normalizar_fecha_spark,normalizar_nombre_columna
from dict_data import dict_transformacion_saps, map_cie_cod


In [0]:
df_pato=spark.table("saps_ctes.bronze.datosctes_consultas_patologia")
df_pato.limit(10).display()

# Realizar las transformaciones y limpieza

In [0]:
#revisamos primero las columnas
df_pato.columns

In [0]:
#de las columnas mostradas arriba, sólo nos interesan las siguientes
col_interes=['ID_SAPS',
 'SAPS',
 'fecha',
 'patologia_desc',
 'agrupacion_CIE10',
 'patologia_cod',
 'id_rango_etario',
 'id_sexo',
 'consulta_cantidad',
 'rango_etario',
 'sexo']
df_pato=df_pato.select(col_interes)
df_pato.columns

In [0]:
#Ahora, revisemos la situación de nuestros datos como los tipos
#o el total de nulos
df_pato.printSchema()

In [0]:

df_pato = df_pato.select([
    col(c).alias(normalizar_nombre_columna(c))
    for c in df_pato.columns
])

In [0]:

#total de nulos
display(df_pato.select([suma(col(c).isNull().cast("int")).alias(c) for c in df_pato.columns])) 

In [0]:
#la cantidad de registro que tenemos
df_pato.count()

# Buscamos los valores de las consultas que no sean enteros

In [0]:
#df_pato.select('consulta_cantidad').distinct().collect()
display(df_pato.filter(~col('consulta_cantidad').rlike(r'^\d+$')).select(['consulta_cantidad']))

# Reemplazamos los valores indicados arriba


In [0]:
reemplazos = {
"|1":"1",
"4.":"4",
"1.":"1",
".30":"30",
".7":"7",
"8+":"8",
".5":"5",
".1":"1",
".2":"2",
"3.":"3",
"4+":"4",
"8+":"8",
"8+4":"12",
"0,6":"6",
"8/":"8",
"5+":"5"
}

# Convertir el diccionario a un mapa de Spark
mapping_expr = create_map([lit(x) for x in chain(*reemplazos.items())])
df_pato = df_pato.withColumn(
    "consulta_cantidad",
    coalesce(mapping_expr.getItem(col("consulta_cantidad")),col("consulta_cantidad")).cast("int")  # Busca en el mapa
)
df_pato.limit(10).display()

Ahora, llenamos los valores nulos con el valor -1 para indicar que son nulos.

In [0]:
df_pato=df_pato.withColumn('consulta_cantidad',when(col('consulta_cantidad').isNull(),-1).otherwise(col('consulta_cantidad')))


# Estandarizar los valores para los sexos

In [0]:
display(df_pato.select('sexo').distinct().collect())
df_pato=df_pato.withColumn(
    "sexo",
    when(col("sexo")=="F","FEMENINO")
    .when(col("sexo")=="M","MASCULINO")
    .otherwise(col("sexo"))
)
display(df_pato.select('sexo').distinct().collect())

# Revisar los valores donde sexo e id_sexo sean nulos

In [0]:
df_pato.filter((col('sexo')=='#N/A') & (col('id_sexo').isNotNull())).display()

Los datos de arriba nos muestran que para los registros donde el sexo es '#N/A' hay valores de id para el sexo, pero estos id difieren de la cantidad de sexos disponibles. Como hay dos sexos, sólo debería haber un id para cada uno. Revisemos cuáles son los id para las mujeres y para los hombres.

In [0]:
df_pato.filter(col('sexo')=='FEMENINO').select('id_sexo').distinct().display()

In [0]:
df_pato.filter(col('sexo')=='MASCULINO').select('id_sexo').distinct().display()

Notamos que para las mujeres hay dos códigos: 1 y 2. Mientras que para los hombres hay sólo 1: el valor 2. Como los valores se solapan, tomamos la decisión de seguir esta regla: 1-FEMENINO, 2-MASCULINO. Realizemos primero ese mapeo.

In [0]:
dict_sex={"FEMENINO":1, "MASCULINO":2}
map_sex=create_map([lit(x) for x in chain (*dict_sex.items())])
df_pato=df_pato.withColumn(
    "id_sexo",
    coalesce(map_sex.getItem(col("sexo")),col("id_sexo")).cast("int")
)

A pesar que la transformacion es correcta, debemos revisar cuáles son los valores que tiene la columna id_sexo de manera más detallada

In [0]:
df_pato.select("id_sexo").distinct().display()

Como vemos, tenemos más valores que 1 y 2. Podemos realizar un mapeo similar al anterior. Ya que dijimos: 1-Femenino y 2-Masculino. Lo que podemos hacer es lo siguiente: aquellos registros que tengan "#N/A" en su sexo, pero tengan 1 en su id_sexo, serán Femenino; y aquellos registros que tengan "#N/A" en su sexo, pero tengan 2 en su id_sexo serán Masculinos.

In [0]:
df_pato=df_pato.withColumn(
    "sexo",
    when((col("id_sexo")==1) & (col("sexo")!="FEMENINO"),"FEMENINO")
    .when((col("id_sexo")==2) & (col("sexo")!="MASCULINO"),"MASCULINO")
    .otherwise(col("sexo"))
)

Lo único que queda ahora es reemplazar los valores "#N/A" por "n/a" para indicar que no hay sexo registrado y cambiar cualquier valor menor a 1 al valor -1 en el campo id_sexo para indicar ausencia de id_sexo.

In [0]:
df_pato=df_pato.withColumn(
    "sexo",
    when((col("sexo")!="FEMENINO") & (col("sexo")!="MASCULINO"),"n/a")
    .otherwise(col("sexo"))
)

In [0]:
df_pato=df_pato.withColumn(
    "id_sexo",
    when((col("id_sexo")<1) | (col("id_sexo")>2) | (col("id_sexo").isNull()),-1)
    .otherwise(col("id_sexo"))
)

# Trabajando con los rangos etarios

In [0]:
df_pato.select(["id_rango_etario","rango_etario"]).distinct().display()

Como vemos en los resultados mostrados, hay situaciones en que varios rangos etarios tienen id repetidos para los mismos rangos etarios. Podemos llegar a la siguiente convención:

- 1	< 1 AÑO
- 2	1 - 4 AÑOS
- 3	5 - 14 AÑOS
- 4	15 - 19 AÑOS
- 5	20-39 AÑOS
- 6	40-69 AÑOS
- 7	>=70 AÑOS

In [0]:
df_pato=df_pato.withColumn(
    "id_rango_etario",
    when(col("rango_etario")=="< 1 AÑO","1")
    .when(col("rango_etario")=="1 - 4 AÑOS","2")
    .when(col("rango_etario")=="5 - 14 AÑOS","3")
    .when(col("rango_etario")=="15 - 19 AÑOS","4")
    .when(col("rango_etario")=="20-39 AÑOS","5")
    .when(col("rango_etario")=="40-69 AÑOS","6")
    .when(col("rango_etario")==">=70 AÑOS","7")
    .otherwise(col("id_rango_etario"))
)

In [0]:
df_pato=df_pato.withColumn(
    "rango_etario",
    when(col("id_rango_etario")=="1","< 1 AÑO")
    .when(col("id_rango_etario")=="2","1 - 4 AÑOS")
    .when(col("id_rango_etario")=="3","5 - 14 AÑOS")
    .when(col("id_rango_etario")=="4","15 - 19 AÑOS")
    .when(col("id_rango_etario")=="5","20-39 AÑOS")
    .when(col("id_rango_etario")=="6","40-69 AÑOS")
    .when(col("id_rango_etario")==">=70 AÑOS",">=70 AÑOS")
    .otherwise(col("rango_etario"))
)


In [0]:
lista_valores_id_sexo=[row.id_rango_etario for row in 
             df_pato.filter(col("rango_etario") != "#N/A")
               .select("id_rango_etario")
               .distinct()
               .collect()]

df_pato = df_pato.withColumn(
    "id_rango_etario",
    when((~col("id_rango_etario").isin(lista_valores_id_sexo)) | (col("id_rango_etario").isNull()), -1)  # condición
    .otherwise(col("id_rango_etario"))  # si no se cumple, deja el valor original
)


Finalmente, reemplazamos los valores '#N/A' en la columna rango etario por el valor 'n/a'.

In [0]:
df_pato=df_pato.withColumn(
    'rango_etario',
    when(col('id_rango_etario')==-1,"n/a")
    .otherwise(col('rango_etario'))
)


# Trabajando con las patologías

Lo primero es revisar si existen registros con códigos de patologia, pero sin descripcion de las patologías. Eso nos permitirá saber qué patologías podemos recuperar su información usando como base el código de las mismas. Para revisar todo el desarrollo de este punto, se sugiere revisar los notebooks ubicados en el siguiente repositorio: https://github.com/eme-19-87/DataAnalytics_Patologias_SAPS_Ctes

In [0]:
#reviso las patologias que tienen código, pero descricipción
df_pato.filter((col("patologia_cod").isNotNull()) & (col("patologia_desc")=="#N/A")).display()

In [0]:
mapping_expr_cie10 = create_map([lit(x) for x in chain(*map_cie_cod.items())])
df_pato = df_pato.withColumn(
    "patologia_desc",
    coalesce(mapping_expr_cie10.getItem(col("patologia_cod")),col("patologia_desc"))
)

Una vez que ya hemos colocado los casos donde tenemos las descripciones de patologías usando el código para llenarlo, podemos buscar el caso contrario: los registros donde tenemos las descripciones de las patologías, pero no tenemos el código.

In [0]:
df_pato.filter((col("patologia_desc").isNotNull()) & (col("patologia_cod")=='#N/A')).display()

No tenemos situaciones como esas. Entonces busquemos aquellos que tengan la agrupación cie10, pero no tenga código de patología.

In [0]:
df_pato.filter((col("agrupacion_CIE10").isNotNull()) & (col("patologia_cod")=='#N/A')).display()

Tampoco existen registros para esos casos. Por lo tanto, completaremos los valores '#N/A' con 'n/a' para las agrupaciones cie10 y las descripciones de patologias.

In [0]:

df_pato = df_pato.withColumn(
    'patologia_desc',
    when(col('patologia_desc')=='#N/A','n/a')
    .otherwise(col('patologia_desc'))
)

df_pato = df_pato.withColumn(
    'agrupacion_CIE10',
    when(col('patologia_desc')=='#N/A','n/a')
    .otherwise(col('agrupacion_CIE10'))
)

Veamos si existen código de patología en nulos.

In [0]:
df_pato.filter(col('patologia_cod').isNull()).display()

Hay registros con valores nulos en el código de patología y, además, no poseen información de la descripción de la patología para conocer el código relacionado. Entonces, reemplacemos esos casos con 'n/a'.

In [0]:
df_pato=df_pato.withColumn(
    'patologia_cod',
    when(col('patologia_cod').isNull(),'n/a')
    .otherwise(col('patologia_desc'))
)

# Trabajando con las fechas

Para el caso de las fechas vamos a completar las fechas nulas con una fecha ficticia que es '1900-01-01' para indicar que no hay fechas presentes. Luego, reformateamos las fechas de tal manera que si están en el formato día, mes, año cambie al formato año-mes-día. Cualquier formato inválido (como por ejemplo sólo mes y año, o sólo mes y día) será tratado como que no existe dicha fecha. Lo mismo sucederá para cualquier otro dato que debería ser fecha y no lo es. Empecemos viendo los valores que posee el campo fecha.

In [0]:
df_pato.select('fecha').distinct().display()

Podemos notar que existen fechas que tienen separadores al caracter '/' y otros que tienen como separador a '-'. Debemos homogeneizar esto. También existen valores nulos, y valores que no tienen completo el formato (por ejemplo, 03-2021).

In [0]:
df_pato = df_pato.withColumn(
    "fecha",
    regexp_replace(col("fecha"), "/", "-")
)


In [0]:
df_pato = df_pato.withColumn(
    "fecha",
    normalizar_fecha_spark(col("fecha"))
)

# Análsiis SAPS e ID_SAPS


Aquí debemos primeramente normalizar los nombres de los saps con los nombres que se encuentra en el archivo 'listado_saps.csv'. Esto es importante para poder buscar una forma de vínculo entre ambas informaciones, porque el listado de saps no tiene los id de los saps. Cosa que si tienen los datos de las patologías. Para ello, ya hemos preparado un diccionario con los datos necesarios para el reemplazo.

In [0]:
df_pato = df_pato.withColumn(
    "id_saps",
    when(col("id_saps").rlike("^[0-9]+$"), col("id_saps"))
    .otherwise(None)
)



In [0]:
mapping_expr_saps = create_map([lit(x) for x in chain(*dict_transformacion_saps.items())])
df_pato = df_pato.withColumn(
    "saps",
    coalesce(mapping_expr_saps.getItem(col("saps")),col("saps"))
)


In [0]:
df_pato.select("saps").distinct().display()

Una vez establecido los nombres de los saps para que se igualen a los nombres que figuran en los datos de la lista de saps, vamos a revisar los id de los saps

In [0]:
df_pato.select("id_saps", "saps").distinct() .orderBy(col("id_saps")) .display()

Vemos que existen casos donde los saps tienen su nombre pero no su código, casos donde hay saps directamente sin código, saps con nombre y en algunos casos tiene código y en otros no. Podemos arreglar eso mediante la creación de un diccionario teniendo en cuenta los nombres de los saps y sus id.

In [0]:
df_pato_saps=df_pato.filter((col('id_saps').isNotNull()) & (col('id_saps')!="#REF!")).select(col("id_saps").alias("id_saps_map"),col("saps")).distinct().orderBy("id_saps")

df_pato = (
    df_pato
    .join(
        broadcast(df_pato_saps),
        on="saps",
        how="left"
    )
    .withColumn(
        "id_saps",
        coalesce(col("id_saps"), col("id_saps_map"))
    )
    .drop("id_saps_map")
)

df_pato.select(['id_saps','saps']).distinct().display()


In [0]:
df_pato.filter((col("id_saps").isNull()) & (col("saps").isNotNull())).select(["id_saps","saps"]).distinct().display()

In [0]:
df_pato.filter(col('saps')=='Dr. Elias Galvan').select(col('id_saps')).distinct().display()

Los datos de arriba notamos que hay saps que no tiene un id asignado. Por lo tanto, podemos asignarle un id nosotros mismos. Una estraegia podría ser crear un diccionario con los nombres de los saps y sus valores serán así:

Primero obtenemos el máximo valor del id_saps y le aumentamos en 1.
Le damos al primer saps ese id y aumentamos en 1.
Repetimos el proceso hasta que los cuatro saps tienen asignado el id

Esta forma es arbitraria, pero no tenemos manera de saber qué id corresponde exactamente.

In [0]:
#casteo primero la columna a enteros
df_pato=df_pato.withColumn(
    "id_saps",
    when(col("id_saps").isNotNull(), col("id_saps").cast('int'))
    .otherwise(col("id_saps"))
)
#obtengo el máximo id para los saps
max_id = (
    df_pato
    .filter((col("id_saps").isNotNull()))
    .select(max("id_saps").alias("max_id").cast("int"))
    .collect()[0]["max_id"]
)

#obtengo los registros sin id_saps pero con saps
df_saps_sin_id = (
    df_pato
    .filter((col("id_saps").isNull()) & (col("saps").isNotNull()))
    .select("saps")
    .distinct()
)

#creo la ventana para aplicar row_number
window = Window.orderBy("saps")

#aplico la función ventana
df_saps_nuevos_ids = (
    df_saps_sin_id
    .withColumn("nuevo_id_saps", row_number().over(window) + lit(max_id))
)

df_saps_sin_id.display()

Hasta este punto tenemos los id para los saps. Pero nos falta completar para el dataframe original. Para ello podemos usar un join

In [0]:
#aplico el join entre dataframes usando el nombre de los saps
#si el id del saps no es nulo, le doy ese id. Si lo es, le doy el nuevo id
df_pato = (
    df_pato
    .join(broadcast(df_saps_nuevos_ids), on="saps", how="left")
    .withColumn("id_saps", when(col("id_saps").isNotNull(), col("id_saps"))
    .otherwise(col("nuevo_id_saps")))
    .drop("nuevo_id_saps")
)

In [0]:
df_pato.select(["id_saps","saps"]).distinct().orderBy(col("saps")).display()

Faltaría llenar los id de saps a -1 si siguen siendo null y con 'n/a' si sigue sieendo null.

In [0]:
df_pato = (
    df_pato
    .withColumn(
        "id_saps",
        when(col("id_saps").isNull(), -1)
        .otherwise(col("id_saps").cast('int'))
    )
    .withColumn(
        "saps",
        when(col("saps").isNull(), "n/a")
        .otherwise(col("saps"))
    )
)

Por último, eliminamos los duplicados.

In [0]:
df_pato = df_pato.dropDuplicates()

# Escribir los resultados en la capa plata

In [0]:
%sql
DROP TABLE IF EXISTS saps_ctes.silver.datosctes_consultas_patologia;

In [0]:
df_pato.write.mode("overwrite").saveAsTable("saps_ctes.silver.datosctes_consultas_patologia")